# Hotel Recommendation Logic
**GitHub repository:** https://github.com/St-virus1210/travel-analytics-capstone

This notebook implements a transparent historical-booking recommender. It combines booking popularity with relative price and optionally boosts hotels/places previously booked by a user. It is not a live availability or pricing service. Place hotels.csv in data/.

In [ ]:
import pandas as pd
hotels=pd.read_csv('data/hotels.csv')
for c in ['price','days','total']: hotels[c]=pd.to_numeric(hotels[c],errors='coerce')
clean=hotels.dropna(subset=['name','place','price']).copy()
stats=clean.groupby(['name','place'],as_index=False).agg(average_price_per_day=('price','mean'),average_days=('days','mean'),average_total=('total','mean'),bookings=('travelCode','count'))
stats['popularity_percentile']=stats['bookings'].rank(pct=True); stats['price_percentile']=stats['average_price_per_day'].rank(pct=True); stats['score']=.65*stats['popularity_percentile']+.35*(1-stats['price_percentile'])
display(stats.sort_values('score',ascending=False).head(10))

def recommend(hotels_frame,user_code=None,place=None,max_price=None,limit=5):
    d=hotels_frame.copy(); d['price']=pd.to_numeric(d['price'],errors='coerce'); d=d.dropna(subset=['name','place','price'])
    if place: d=d[d['place'].astype(str).str.contains(str(place),case=False,na=False)]
    if max_price is not None: d=d[d['price']<=float(max_price)]
    if d.empty: return pd.DataFrame()
    agg=d.groupby(['name','place'],as_index=False).agg(average_price_per_day=('price','mean'),average_days=('days','mean'),average_total=('total','mean'),bookings=('travelCode','count'))
    agg['score']=.65*agg['bookings'].rank(pct=True)+.35*(1-agg['average_price_per_day'].rank(pct=True))
    if user_code is not None:
        history=hotels_frame[pd.to_numeric(hotels_frame['userCode'],errors='coerce')==int(user_code)]
        agg.loc[agg['place'].isin(history['place'].dropna()),'score']+=.25; agg.loc[agg['name'].isin(history['name'].dropna()),'score']+=.35
    return agg.sort_values(['score','bookings'],ascending=False).head(max(1,min(int(limit),50)))
display(recommend(hotels,user_code=0,limit=5))
print('For rigorous evaluation, use a time-aware holdout and report Precision@K, Recall@K and catalog coverage.')